In [4]:
# Compute per-dimension mean/std over a few episodes and build Normalizer for batched data
import minari
import numpy as np
import torch
from functools import partial
from torch.utils.data import DataLoader

from utilities.dataset_utils import (
    collate_fn,
    compute_stats_over_episodes,
    create_normalizers_from_stats,
    normalize_batch_dict,
)

# Load dataset (adjust if needed)
dataset_name = 'mujoco/halfcheetah/simple-v0'
dataset = minari.load_dataset(dataset_name, download=True)

# Compute stats over the first N episodes (e.g., 5)
N_EPISODES = 20
stats = compute_stats_over_episodes(dataset, keys=("observations",), num_episodes=N_EPISODES)
print("observations mean (first 5 dims):", stats["observations"]["mean"][:5])
print("observations std  (first 5 dims):", stats["observations"]["std"][:5])

# Build normalizers and get shift/scale if needed for env wrappers
normalizers = create_normalizers_from_stats(stats)
obs_shift, obs_scale = normalizers["observations"].shift_scale()
print("shift[:5]", obs_shift[:5], "scale[:5]", obs_scale[:5])

# Build a dataloader producing collated windows/batches
loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=partial(collate_fn, shuffle_trajectories=False),
    num_workers=0,
    pin_memory=False,
)

batch = next(iter(loader))
print("raw batch observations shape:", tuple(batch["observations"].shape))

# Apply normalization to selected keys
norm_batch = normalize_batch_dict(batch, normalizers, keys=("observations", "next_observations"))

# Quick sanity check: mean/std over the last dimension for the first few items
x = norm_batch["observations"].float()  # [pad, T, B, D]
# Flatten leading dims except feature dim
x_flat = x.reshape(-1, x.shape[-1])
mean_check = x_flat.mean(dim=0).cpu().numpy()
std_check = x_flat.std(dim=0, unbiased=False).cpu().numpy()
print("normalized mean:", np.round(mean_check, 4))
print("normalized std:", np.round(std_check, 4))



observations mean (first 5 dims): [-0.04722835  0.06937508  0.11161757 -0.13164426  0.02210294]
observations std  (first 5 dims): [0.04009089 0.13611725 0.48063663 0.28526977 0.35577002]
shift[:5] [ 0.04722835 -0.06937508 -0.11161757  0.13164426 -0.02210294] scale[:5] [24.942705   7.3465533  2.0805695  3.5054414  2.8107965]
raw batch observations shape: (1, 30, 10, 32, 17)
normalized mean: [ 0.008   0.0193  0.0016  0.0152  0.0119 -0.0031 -0.012   0.0113 -0.0259
 -0.0008 -0.0052 -0.0082  0.0137 -0.0104  0.0021 -0.0004  0.0016]
normalized std: [1.0237 0.9872 1.0036 1.0077 1.0019 1.0071 1.0007 0.9867 0.974  1.0193
 0.9959 1.0097 1.0115 0.9852 1.0166 0.9975 0.9807]


In [71]:
import minari
dataset_simple = minari.load_dataset('mujoco/halfcheetah/simple-v0', download=True)

In [68]:
dataset_medium = minari.load_dataset('mujoco/halfcheetah/medium-v0', download=True)
dataset_expert = minari.load_dataset('mujoco/halfcheetah/expert-v0', download=True)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

halfcheetah/medium-v0/data/main_data.hdf(…):   0%|          | 0.00/210M [00:00<?, ?B/s]


Dataset mujoco/halfcheetah/medium-v0 downloaded to /home/haitong/.minari/datasets/mujoco/halfcheetah/medium-v0



Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

halfcheetah/expert-v0/data/main_data.hdf(…):   0%|          | 0.00/210M [00:00<?, ?B/s]


Dataset mujoco/halfcheetah/expert-v0 downloaded to /home/haitong/.minari/datasets/mujoco/halfcheetah/expert-v0


In [72]:
dataset_list = [dataset_simple, dataset_medium, dataset_expert]
for ep1, ep2, ep3 in zip(*dataset_list):
    print(ep1.observations.shape, ep2.observations.shape, ep3.observations.shape)
    break


(1001, 17) (1001, 17) (1001, 17)


In [4]:
import torch
for episode_data in dataset.iterate_episodes():
    observations = episode_data.observations
    observations = torch.tensor(observations)
    mean = torch.mean(observations, dim=0)
    std = torch.std(observations, dim=0)
    print(mean, std)
    break

tensor([-0.0474,  0.0865,  0.1013, -0.1300,  0.0291,  0.4229, -0.0953,  0.1426,
         7.2454, -0.0688, -0.0709,  0.1174, -0.2665,  0.0515, -0.1087,  0.4010,
         0.1697], dtype=torch.float64) tensor([ 0.0430,  0.1536,  0.4754,  0.2961,  0.3564,  0.1908,  0.5054,  0.3171,
         1.3242,  0.5479,  1.5876, 10.8264,  5.9853,  8.6909,  4.8870, 12.9113,
         7.9878], dtype=torch.float64)


In [3]:
env = dataset.recover_environment()
env.action_space.shape[0]

/home/haitong/anaconda3/envs/drones/lib/python3.10/site-packages/minari/dataset/minari_dataset.py:204: UserWarning: Installed mujoco version 3.3.0 does not meet the requirement ==3.2.3.
We recommend to install the required version with `pip install "mujoco==3.2.3"`
  warnings.warn(


6

In [4]:
env = dataset.recover_environment()

In [6]:
env.reset()

(array([-0.03019876, -0.0508219 ,  0.06213242,  0.09357072,  0.09359573,
        -0.08075633, -0.07568326, -0.09290087, -0.04226936,  0.08356013,
         0.0027152 , -0.05849673, -0.11837438,  0.22317824,  0.15275628,
        -0.14830531, -0.04747396]),
 {'x_position': -0.030782340129538996})

In [5]:
import torch
import torch.nn.utils.rnn as rnn_utils
from torch.utils.data import DataLoader
import numpy as np

def collate_fn(batch, shuffle_trajectories=False):
    def map_fn(x):
        if shuffle_trajectories:
            return torch.as_tensor(np.random.permutation(x))
        else:
            return torch.as_tensor(x)
    return {
        "id": torch.Tensor([x.id for x in batch]),
        "observations": torch.nn.utils.rnn.pad_sequence(
            [map_fn(x.observations) for x in batch],
            batch_first=True
        ),
        "actions": torch.nn.utils.rnn.pad_sequence(
            [map_fn(x.actions) for x in batch],
            batch_first=True
        ),
        "rewards": torch.nn.utils.rnn.pad_sequence(
            [map_fn(x.rewards) for x in batch],
            batch_first=True
        ),
        "terminations": torch.nn.utils.rnn.pad_sequence(
            [map_fn(x.terminations) for x in batch],
            batch_first=True
        ),
        "truncations": torch.nn.utils.rnn.pad_sequence(
            [map_fn(x.truncations) for x in batch],
            batch_first=True
        )
    }

from functools import partial

dataloader = DataLoader(
    dataset, 
    batch_size=1, 
    shuffle=True, 
    collate_fn=partial(collate_fn, shuffle_trajectories=False),
    num_workers=4, 
    pin_memory=True # Set to True if you are training on a CUDA GPU
)

In [15]:
from torch.nn import functional as F
n = 5
for batch in dataloader:
    print(batch['observations'].shape)
    pad = (0, 0, 0, n)
    rewards_pad = F.pad(batch['observations'], pad, value=0.0)
    print(rewards_pad[0, -10:])
    break

torch.Size([1, 1001, 17])
tensor([[-4.8847e-03, -3.0981e-02,  8.4126e-01, -3.5313e-01,  1.9033e-01,
          4.9019e-01, -5.3826e-01, -1.9138e-01,  7.3602e+00, -4.4081e-01,
          5.6125e-02,  2.3377e+00,  2.3708e+00, -1.1243e+01, -1.9827e-01,
          6.2134e-01, -1.2773e+01],
        [-2.8958e-02,  3.0589e-02,  7.1275e-01, -3.1377e-01, -1.0376e-01,
          5.3309e-01, -5.5216e-01, -5.5396e-01,  7.1268e+00, -5.1988e-01,
          1.7506e+00, -4.6840e+00, -7.1442e-01, -1.9862e+00,  1.4275e+00,
         -7.9523e-01,  1.0910e+00],
        [-5.2772e-02,  1.3986e-01, -7.7079e-02, -1.1691e-01, -3.2267e-01,
          4.1231e-01, -1.0988e-01, -1.0516e-01,  6.1299e+00, -8.6290e-01,
          1.7571e+00, -1.8153e+01,  2.4073e+00, -4.8318e+00, -3.0457e+00,
          1.2752e+01,  1.2353e+01],
        [-7.1299e-02,  7.0516e-02, -5.8603e-01, -1.4466e-02, -4.2637e-01,
          1.6107e-01,  6.7580e-01,  5.6845e-01,  7.2489e+00,  1.4040e-01,
         -1.4318e+00,  1.5759e+00, -2.5918e+00,  4.1

In [19]:
a = np.arange(5)
a

array([0, 1, 2, 3, 4])

In [24]:
a[:5]

array([0, 1, 2, 3, 4])

# Create stride of the dataset

In [62]:
obs = batch['observations']
obs = obs[0].numpy()
obs.strides

(136, 8)

In [49]:
import numpy as np
from numpy.lib.stride_tricks import as_strided

# --- Parameters ---
L = 1000       # Original length
dim = 64       # Feature dimension (example)
T = 50         # Sub-trajectory length
S = 1          # Stride (1 for full overlap)
B = 32         # Batch size

# --- Example Data ---
# Your data: shape (1000, dim)
data = np.arange(L * dim).reshape(L, dim)

# --- 1. Create all overlapping windows (as a view) ---

# Get strides of original array (bytes per step in each dim)
stride_L, stride_dim = data.strides

# Calculate number of windows
num_windows = (L - T) // S + 1

# Define the shape of the new view
shape = (num_windows, T, dim)

# Define the strides of the new view
strides = (S * stride_L, stride_L, stride_dim)

# Create the view
# shape: (num_windows, T, dim)
all_samples = as_strided(data, shape=shape, strides=strides)

# --- 2. Batch the windows ---

# Calculate how many full batches we can make (drops remainders)
num_batches = num_windows // B
N_usable = num_batches * B

# Take a slice of the view (this is still a view)
# shape: (N_usable, T, dim)
usable_samples = all_samples[:N_usable]

# Reshape into batches
# shape: (num_batches, B, T, dim)
batched_data = usable_samples.reshape(num_batches, B, T, dim)

# --- 3. Permute to [T, B, dim] ---

# Transpose to get your desired [T, B, dim] format per batch
# shape: (num_batches, T, B, dim)
final_batches = batched_data.transpose(0, 2, 1, 3)

# If you need a new array in memory, make a copy
# final_batches_copy = final_batches.copy()

print(f"Original data shape: {data.shape}")
print(f"Final batched data shape: {final_batches.shape}")
print(f"Shape of one batch: {final_batches[0].shape}")
print(final_batches[0, :, 0, :])

Original data shape: (1000, 64)
Final batched data shape: (29, 50, 32, 64)
Shape of one batch: (50, 32, 64)
[[   0    1    2 ...   61   62   63]
 [  64   65   66 ...  125  126  127]
 [ 128  129  130 ...  189  190  191]
 ...
 [3008 3009 3010 ... 3069 3070 3071]
 [3072 3073 3074 ... 3133 3134 3135]
 [3136 3137 3138 ... 3197 3198 3199]]


In [56]:
T = 50
S = 2
num_windows = (1000 - T) // S + 1  # 951

new_shape = (num_windows, T, data.shape[1])  # (951, 50, 64)
print(new_shape)

# Original strides
stride_L, stride_dim = data.strides  # (512, 8)

new_strides = (S * stride_L, stride_L, stride_dim)
# (1 * 512,  512,    8)
# (512,      512,    8)
print(new_strides)

view = np.lib.stride_tricks.as_strided(data,
                                       shape=new_shape,
                                       strides=new_strides)

(476, 50, 64)
(1024, 512, 8)


What is happening in the new `view`?

The new view has `shape=(951, 50, 64)` and `strides=(512, 512, 8)`.

Let's see how it finds an element, like `view[1, 0, 0]`:

Start at the beginning of the original data buffer (Byte 0).

- Move along Axis 0: `1 * strides[0]` = `1 * 512 bytes`.

- Move along Axis 1: `0 * strides[1]` = `0 * 512 bytes`.

- Move along Axis 2: `0 * strides[2]` = `0 * 8 bytes`.

- Total offset = 512 bytes.

- This points to `data[1, 0]`. This is correct! `view[1, 0, 0]` should be the start of the second window, which starts at index 1 of the original data.

Now let's find `view[0, 1, 0]`:

- Start at `Byte 0`.

- Move along Axis 0: `0 * 512 bytes`.

- Move along Axis 1: `1 * 512 bytes`.

- Move along Axis 2: `0 * 8 bytes`.

Total offset = `512 bytes`.

This also points to `data[1, 0]`.

This shows the memory overlap. view[1, 0, 0] and view[0, 1, 0] are the exact same piece of memory. This is why as_strided is so efficient: no data is copied. The new array is just a clever set of instructions for "hopping" around the original data buffer.

In [60]:
from numpy.lib.stride_tricks import sliding_window_view

# Your (L, dim) data
data = np.arange(1000 * 64).reshape(1000, 64)

# Create windows of size T=50 along axis 0
# The result has shape (951, 64, 50)
view_safe = sliding_window_view(data, window_shape=(50, ), axis=(0, ))

print(view_safe.shape)

(951, 64, 50)


In [66]:
import torch

# --- Parameters ---
L = 1000       # Original length
dim = 64       # Feature dimension (example)
T = 50         # Sub-trajectory length
S = 2          # Stride (1 for full overlap)
B = 32         # Batch size

# --- Example Data ---
# Your data: shape (1000, dim)
data = torch.arange(L * dim).reshape(L, dim)

# --- 1. Create all overlapping windows ---

# .unfold(dimension, size, step)
# Note: PyTorch puts the new dimension (T) last
# shape: (num_windows, dim, T)
all_samples_torch_format = data.unfold(dimension=0, size=T, step=S)
print(all_samples_torch_format.shape)

# Permute to get (num_windows, T, dim)
# shape: (num_windows, T, dim)
all_samples = all_samples_torch_format.permute(0, 2, 1)

# --- 2. Batch the windows ---

num_windows = all_samples.shape[0]
num_batches = num_windows // B
N_usable = num_batches * B

# shape: (N_usable, T, dim)
usable_samples = all_samples[:N_usable]

# Reshape into batches
# shape: (num_batches, B, T, dim)
batched_data = usable_samples.view(num_batches, B, T, dim)

# --- 3. Permute to [T, B, dim] ---

# Transpose to get your desired [T, B, dim] format per batch
# shape: (num_batches, T, B, dim)
final_batches = batched_data.permute(0, 2, 1, 3)

print(f"Original data shape: {data.shape}")
print(f"Final batched data shape: {final_batches.shape}")
print(f"Shape of one batch: {final_batches[0].shape}")

torch.Size([476, 64, 50])
Original data shape: torch.Size([1000, 64])
Final batched data shape: torch.Size([14, 50, 32, 64])
Shape of one batch: torch.Size([50, 32, 64])
